### MLP-Mixer: image classification with NNTile

MLP-Mixer is a vision architecture without self-attention. An image is split into
fixed-size patches; each patch is a token. A **Mixer** block alternates:

- **Token-mixing MLP** (along the patch axis),
- **Channel-mixing MLP** (along the feature axis),

with residual connections and LayerNorm, as in the reference PyTorch model in
`nntile.torch_models.mlp_mixer`.

The full NNTile model is: linear projection of patch vectors → stack
of **MixerBlock** modules → **GAP** (global average pooling over patches) →
**classifier** (linear head).

### 1. Environment variable setting block

The following block sets environment variables read during training. Change them
between runs if needed (GPU id, StarPU workers, etc.).


In [1]:
# Preliminary setup of experimental environment
import os
from pathlib import Path

nntile_dir = Path.cwd() / ".."

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["PYTHONPATH"] = str(
    nntile_dir / "build" / "wrappers" / "python"
)

os.environ["STARPU_NCPU"] = "1"
os.environ["STARPU_NCUDA"] = "1"
os.environ["STARPU_SILENT"] = "1"
os.environ["STARPU_SCHED"] = "dmdasd"
os.environ["STARPU_FXT_TRACE"] = "0"
os.environ["STARPU_WORKERS_NOBIND"] = "1"
os.environ["STARPU_PROFILING"] = "1"
os.environ["STARPU_BUS_STATS"] = "1"
os.environ["STARPU_HOME"] = str(Path.cwd() / "starpu")
os.environ["STARPU_PERF_MODEL_DIR"] = str(
    Path(os.environ["STARPU_HOME"]) / "sampling"
)
os.environ["STARPU_PERF_MODEL_HOMOGENEOUS_CPU"] = "1"
os.environ["STARPU_PERF_MODEL_HOMOGENEOUS_CUDA"] = "1"
os.environ["STARPU_HOSTNAME"] = "MLP_Mixer_example"
os.environ["STARPU_FXT_PREFIX"] = str(
    Path(os.environ["STARPU_HOME"]) / "fxt"
)


### 2. Data and model layout

Inside `mlp_mixer_training.py` the script downloads a torchvision dataset into `--data-root`, patches images to `[n_patches, minibatch, patch_dim]`, and builds StarPU batches.

Grayscale 28×28 (**MNIST**, **Fashion-MNIST**): `patch_size=7`. **CIFAR-10** (RGB 32×32): `patch_size=4`.

Batch count per epoch: `floor(dataset_size / batch_size)`. Examples below use **50** steps for MNIST/Fashion (`batch_size=1200`) and the same idea for CIFAR (`batch_size=1000`, 50 000 images).


### 3. Example scenarios (`mlp_mixer_training.py`)

- **Random init + save** (3.1 MNIST, 3.3 Fashion-MNIST, 3.5 CIFAR-10)
- **Resume checkpoint** (3.2 MNIST fp32)
- **Resume with TF32** (3.2b)
- **Resume with bf16** (3.2c, 3.4 Fashion-MNIST)

#### 3.1. Train from random initialization (MNIST)

Notebook defaults: `--batch-size 1200`, `--minibatch-size 60` → **50** optimizer steps per epoch on MNIST/Fashion-MNIST (60 000 / 1200). Peak GPU memory is set by minibatch size, not full batch.


In [2]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype fp32 \
    --restrict cuda \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1 \
    --save-checkpoint-path .model/mlp_mixer_mnist.pt

Namespace(dataset='mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='', save_checkpoint_path='.model/mlp_mixer_mnist.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='fp32', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
MlpMixer(
  (mixer_sequence): Sequential(
    (0): Linear(in_features=49, out_features=512, bias=False)
    (1): Mixer(
      (norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_1): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=False)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=16, bias=False)
        )
      )
      (norm_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_2): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): GELU(approximate=

#### 3.2. Resume training (fp32, same dtype as 3.1)

Same architecture flags and `--dtype fp32` as the run that created the checkpoint.


In [4]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype fp32 \
    --restrict cuda \
    --checkpoint-path .model/mlp_mixer_mnist.pt \
    --save-checkpoint-path .model/mlp_mixer_mnist_resume.pt \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1

Namespace(dataset='mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='.model/mlp_mixer_mnist.pt', save_checkpoint_path='.model/mlp_mixer_mnist_resume.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='fp32', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
/home/g.karpov/nntile/notebooks/../wrappers/python/examples/mlp_mixer_training.py:115: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be al

#### 3.2b. Resume with TF32

Load the checkpoint from 3.1 but train in **`tf32`** (`Tensor_fp32_fast_tf32`). Requires CUDA (`--restrict cuda`). Same flags; only `--dtype` changes.


In [2]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype tf32 \
    --restrict cuda \
    --checkpoint-path .model/mlp_mixer_mnist.pt \
    --save-checkpoint-path .model/mlp_mixer_mnist_tf32.pt \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1

Namespace(dataset='mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='.model/mlp_mixer_mnist.pt', save_checkpoint_path='.model/mlp_mixer_mnist_tf32.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='tf32', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
/home/g.karpov/nntile/notebooks/../wrappers/python/examples/mlp_mixer_training.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allo

#### 3.2c. Resume with bf16

Same checkpoint as 3.1; **`--restrict cuda` is mandatory** for `Tensor_bf16`.


In [3]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype bf16 \
    --restrict cuda \
    --checkpoint-path .model/mlp_mixer_mnist.pt \
    --save-checkpoint-path .model/mlp_mixer_mnist_bf16.pt \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1

Namespace(dataset='mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='.model/mlp_mixer_mnist.pt', save_checkpoint_path='.model/mlp_mixer_mnist_bf16.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='bf16', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
/home/g.karpov/nntile/notebooks/../wrappers/python/examples/mlp_mixer_training.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allo

#### 3.3. Fashion-MNIST: train from scratch

Same flags as MNIST (3.1): 50 batches per epoch.


In [5]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset fashion_mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype fp32 \
    --restrict cuda \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1 \
    --save-checkpoint-path .model/mlp_mixer_fashion_mnist.pt

Namespace(dataset='fashion_mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='', save_checkpoint_path='.model/mlp_mixer_fashion_mnist.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='fp32', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
MlpMixer(
  (mixer_sequence): Sequential(
    (0): Linear(in_features=49, out_features=512, bias=False)
    (1): Mixer(
      (norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_1): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=False)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=16, bias=False)
        )
      )
      (norm_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_2): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): G

#### 3.4. Fashion-MNIST: resume from checkpoint with bf16

Load weights from 3.3; same `--dataset` and architecture flags. **`--restrict cuda` is mandatory** for `Tensor_bf16`.


In [4]:
!mkdir -p .model
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset fashion_mnist \
    --data-root .data \
    --batch-size 1200 \
    --minibatch-size 60 \
    --patch-size 7 \
    --hidden-dim 512 \
    --num-mixer-layers 4 \
    --dtype bf16 \
    --restrict cuda \
    --checkpoint-path .model/mlp_mixer_fashion_mnist.pt \
    --save-checkpoint-path .model/mlp_mixer_fashion_mnist_resume_bf16.pt \
    --optimizer adam \
    --lr 1e-4 \
    --nepochs 1

Namespace(dataset='fashion_mnist', data_root='.data', batch_size=1200, minibatch_size=60, patch_size=7, hidden_dim=512, num_mixer_layers=4, checkpoint_path='.model/mlp_mixer_fashion_mnist.pt', save_checkpoint_path='.model/mlp_mixer_fashion_mnist_resume_bf16.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='bf16', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
/home/g.karpov/nntile/notebooks/../wrappers/python/examples/mlp_mixer_training.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary

#### 3.5. CIFAR-10

RGB 32×32; use `--patch-size 4`.


In [7]:
!python3 ../wrappers/python/examples/mlp_mixer_training.py \
    --dataset cifar10 --data-root .data --batch-size 1000 --minibatch-size 50 \
    --patch-size 4 --hidden-dim 512 --num-mixer-layers 4 --dtype fp32 \
    --restrict cuda --save-checkpoint-path .model/mlp_mixer_cifar10.pt --nepochs 1

Namespace(dataset='cifar10', data_root='.data', batch_size=1000, minibatch_size=50, patch_size=4, hidden_dim=512, num_mixer_layers=4, checkpoint_path='', save_checkpoint_path='.model/mlp_mixer_cifar10.pt', optimizer='adam', lr=0.0001, nepochs=1, dtype='fp32', restrict='cuda', logger=False, logger_server_addr='localhost', logger_server_port=5001)
MlpMixer(
  (mixer_sequence): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=False)
    (1): Mixer(
      (norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_1): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=False)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=False)
        )
      )
      (norm_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp_2): MixerMlp(
        (fn): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): GELU(approx